In [3]:
import urllib.request
import os
import datetime
import pandas as pd

print("Все супер! Бібліотеки підключено.")

Все супер! Бібліотеки підключено.


In [4]:
# Комірка 2: Завантаження даних
def download_vhi_data(output_dir="vhi_data"):
    # Створюємо папку для збереження файлів, якщо її ще немає
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Створено папку: {output_dir}")

    # Отримуємо поточний рік автоматично
    current_year = datetime.datetime.now().year

    # В базі NOAA Україна має 27 адміністративних індексів (від 1 до 27)
    for province_id in range(1, 28):
        # Перевіряємо, чи вже є збережений файл для цієї області (запобігання колізії)
        existing_files = [f for f in os.listdir(output_dir) if f.startswith(f"vhi_id_{province_id}_")]
        
        if existing_files:
            print(f"Область {province_id}: дані вже існують. Пропускаємо.")
            continue
            
        # Формуємо посилання для завантаження
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2={current_year}&type=Mean"
        
        try:
            # Завантажуємо дані
            wp = urllib.request.urlopen(url)
            text = wp.read()
            
            # Створюємо мітку часу для назви файлу (рік-місяць-день_година-хвилина-секунда)
            now = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            filename = f"vhi_id_{province_id}_{now}.csv"
            filepath = os.path.join(output_dir, filename)
            
            # Зберігаємо завантажений текст у файл
            with open(filepath, 'wb') as f:
                f.write(text)
                
            print(f"Область {province_id}: завантажено ({filename})")
            
        except Exception as e:
            print(f"Помилка для області {province_id}: {e}")

# Викликаємо функцію, щоб вона почала працювати
download_vhi_data()

Область 1: дані вже існують. Пропускаємо.
Область 2: дані вже існують. Пропускаємо.
Область 3: дані вже існують. Пропускаємо.
Область 4: дані вже існують. Пропускаємо.
Область 5: дані вже існують. Пропускаємо.
Область 6: дані вже існують. Пропускаємо.
Область 7: дані вже існують. Пропускаємо.
Область 8: дані вже існують. Пропускаємо.
Область 9: дані вже існують. Пропускаємо.
Область 10: дані вже існують. Пропускаємо.
Область 11: дані вже існують. Пропускаємо.
Область 12: дані вже існують. Пропускаємо.
Область 13: дані вже існують. Пропускаємо.
Область 14: дані вже існують. Пропускаємо.
Область 15: дані вже існують. Пропускаємо.
Область 16: дані вже існують. Пропускаємо.
Область 17: дані вже існують. Пропускаємо.
Область 18: дані вже існують. Пропускаємо.
Область 19: дані вже існують. Пропускаємо.
Область 20: дані вже існують. Пропускаємо.
Область 21: дані вже існують. Пропускаємо.
Область 22: дані вже існують. Пропускаємо.
Область 23: дані вже існують. Пропускаємо.
Область 24: дані вже

In [5]:
# Комірка 3: Очищення даних (Data Cleaning)
def clean_and_load_data(data_dir="vhi_data"):
    all_data = []
    
    # Словник для заміни індексів (NOAA ID -> (Новий ID, Назва області українською))
    index_mapping = {
        1: (22, 'Черкаська'), 2: (24, 'Чернігівська'), 3: (23, 'Чернівецька'),
        4: (25, 'Республіка Крим'), 5: (3, 'Дніпропетровська'), 6: (4, 'Донецька'),
        7: (8, 'Івано-Франківська'), 8: (19, 'Харківська'), 9: (20, 'Херсонська'),
        10: (21, 'Хмельницька'), 11: (9, 'Київська'), 12: (26, 'Київ (місто)'),
        13: (10, 'Кіровоградська'), 14: (11, 'Луганська'), 15: (12, 'Львівська'),
        16: (13, 'Миколаївська'), 17: (14, 'Одеська'), 18: (15, 'Полтавська'),
        19: (16, 'Рівненська'), 20: (27, 'Севастополь (місто)'), 21: (17, 'Сумська'),
        22: (18, 'Тернопільська'), 23: (6, 'Закарпатська'), 24: (1, 'Вінницька'),
        25: (2, 'Волинська'), 26: (7, 'Запорізька'), 27: (5, 'Житомирська')
    }
    
    # Перебираємо всі скачані файли
    for filename in os.listdir(data_dir):
        if not filename.endswith(".csv"): 
            continue
            
        # Витягуємо старий ID з назви файлу (наприклад, з vhi_id_1_... витягне 1)
        old_id = int(filename.split('_')[2])
        filepath = os.path.join(data_dir, filename)
        
        # Зчитуємо файл. header=1 пропускає перший рядок, names задає правильні назви колонок
        headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']
        df = pd.read_csv(filepath, header=1, names=headers)
        
        # 1. Видаляємо останній порожній стовпець
        df = df.drop(columns=['empty'], errors='ignore')
        
        # 2. Видаляємо рядки, де всі значення порожні
        df = df.dropna()
        
        # 3. Видаляємо рядки з відсутніми даними (в NOAA це позначається як -1)
        df = df[df['VHI'] != -1]
        
        # 4. Очищуємо стовпець Year від HTML-тегів та перетворюємо на числа
        df['Year'] = df['Year'].astype(str).str.replace('<tt><pre>', '').str.replace('</pre></tt>', '')
        df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
        df = df.dropna(subset=['Year'])
        df['Year'] = df['Year'].astype(int)
        
        # 5. Додаємо нові стовпці з правильною назвою та індексом області
        new_id, region_name = index_mapping.get(old_id, (old_id, "Невідомо"))
        df['Region_Name'] = region_name
        df['Region_ID'] = new_id
            
        all_data.append(df)
        
    # Зліплюємо всі 27 таблиць в одну велику
    full_df = pd.concat(all_data, ignore_index=True)
    return full_df

# Запускаємо нашу функцію і зберігаємо результат у змінну df
df = clean_and_load_data()

# Виводимо перші 5 рядків, щоб подивитися, яка краса вийшла
df.head()

,Year,Week,SMN,SMT,VCI,TCI,VHI,Region_Name,Region_ID
0,1982,1.0,0.059,258.24,51.11,48.78,49.95,Хмельницька,21
1,1982,2.0,0.063,261.53,55.89,38.20,47.04,Хмельницька,21
2,1982,3.0,0.063,263.45,57.30,32.69,44.99,Хмельницька,21
3,1982,4.0,0.061,265.10,53.96,28.62,41.29,Хмельницька,21
4,1982,5.0,0.058,266.42,46.87,28.57,37.72,Хмельницька,21


### Завдання 1: Ряд VHI для області за вказаний рік

In [6]:
def get_vhi_by_region_and_year(dataframe, region_id, year):
    # Фільтруємо таблицю за ID області та роком
    filtered_df = dataframe[(dataframe['Region_ID'] == region_id) & (dataframe['Year'] == year)]
    # Повертаємо лише колонки з тижнем та VHI
    return filtered_df[['Week', 'VHI']]

# Викликаємо функцію: наприклад, для Вінницької області (ID 1) за 2005 рік
print("VHI для Вінницької області (ID 1) за 2005 рік:")
display(get_vhi_by_region_and_year(df, 1, 2005).head())

VHI для Вінницької області (ID 1) за 2005 рік:


,Week,VHI
34837,1.0,56.65
34838,2.0,55.96
34839,3.0,56.88
34840,4.0,58.05
34841,5.0,58.71


### Завдання 2: Ряд VHI за вказаний діапазон років для вказаних областей

In [7]:
def get_vhi_by_years_and_regions(dataframe, region_ids, start_year, end_year):
    # Фільтруємо за списком областей та діапазоном років
    filtered_df = dataframe[
        (dataframe['Region_ID'].isin(region_ids)) & 
        (dataframe['Year'] >= start_year) & 
        (dataframe['Year'] <= end_year)
    ]
    return filtered_df[['Year', 'Week', 'VHI', 'Region_Name']]

# Викликаємо: для Київської (9) та Львівської (12) областей з 2010 по 2011 рік
print("VHI для Київської та Львівської областей (2010-2011):")
display(get_vhi_by_years_and_regions(df, [9, 12], 2010, 2011).head(10))

VHI для Київської та Львівської областей (2010-2011):


,Year,Week,VHI,Region_Name
3652,2010,1.0,47.84,Київська
3653,2010,2.0,47.23,Київська
3654,2010,3.0,49.70,Київська
3655,2010,4.0,52.06,Київська
3656,2010,5.0,52.79,Київська
3657,2010,6.0,52.00,Київська
3658,2010,7.0,50.24,Київська
3659,2010,8.0,48.74,Київська
3660,2010,9.0,47.40,Київська
3661,2010,10.0,45.59,Київська


### Завдання 3: Пошук екстремумів (min та max), середнього та медіани

In [8]:
def get_vhi_statistics(dataframe, region_id, year):
    # Беремо дані тільки для конкретної області і року
    subset = dataframe[(dataframe['Region_ID'] == region_id) & (dataframe['Year'] == year)]
    
    # Рахуємо статистику
    stats = {
        'Рік': year,
        'Область_ID': region_id,
        'Мінімум VHI': subset['VHI'].min(),
        'Максимум VHI': subset['VHI'].max(),
        'Середнє VHI': round(subset['VHI'].mean(), 2),
        'Медіана VHI': subset['VHI'].median()
    }
    
    # Виводимо як гарну табличку (DataFrame)
    return pd.DataFrame([stats])

# Викликаємо: для Одеської області (14) за 2015 рік
print("Статистика VHI для Одеської області за 2015 рік:")
display(get_vhi_statistics(df, 14, 2015))

Статистика VHI для Одеської області за 2015 рік:


,Рік,Область_ID,Мінімум VHI,Максимум VHI,Середнє VHI,Медіана VHI
0,2015,14,18.41,59.08,38.81,40.23
